# 从零完成 Project 3：Mean-Reverting Stock Trading with RL

这个 notebook 是最终版项目骨架：它既是作业实现，也是一个面向第一次做 RL 实践的学习笔记。

核心问题是：股票对数收益率服从均值回复过程，交易者选择连续持仓 \(A_t\)，用强化学习学习交易策略，并比较不同数据协议与算法。

本 notebook 覆盖：
- RL 入门：state/action/reward/policy/value、on-policy/off-policy。
- 作业模型：均值回复价格、两种 reward、oracle vs sequential data。
- 理论基准：Reward (1) 的解析最优策略。
- 算法：PPO、TD3、SAC 的 from-scratch PyTorch 实现。
- 实验：R1/R2、oracle/sequential、PPO/TD3/SAC、policy visualization、learning-rate ablation、full-mode parameter sensitivity。

默认 `RUN_FULL = False`，用于快速从头运行和检查代码。把 `RUN_FULL = True` 后会执行更完整、耗时更长的重实验。


In [ ]:
import math
import random
from dataclasses import dataclass, replace
from collections import deque
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
from IPython.display import display

DEVICE = torch.device("cpu")
pd.set_option("display.max_columns", 80)
plt.rcParams["figure.dpi"] = 120


## 0. 运行模式与实验预算

`RUN_FULL=False` 时，本 notebook 是 fast smoke run：每个算法只训练少量步数，目标是验证实现、生成表格和图，并帮助理解流程。

`RUN_FULL=True` 时，预算会明显变大：
- 更多训练步数。
- 更多 Monte Carlo evaluation paths。
- 多 seed。
- 多组 \((\kappa,\sigma,\lambda,\gamma)\) 参数敏感性。

作业提交前建议先用 fast mode 确认没有错误，再开启 full mode 跑最终数字。


In [ ]:
RUN_FULL = False
RUN_MAIN_MATRIX = True
RUN_ABLATION = True
RUN_SENSITIVITY = RUN_FULL

@dataclass
class MarketConfig:
    kappa: float = 0.2
    sigma: float = 0.1
    lam: float = 0.01
    gamma: float = 0.95
    a_max: float = 10.0
    T: int = 80
    log_s_clip: Optional[float] = None

@dataclass
class Budget:
    hidden: int
    n_eval: int
    seeds: List[int]
    sequential_path_len: int
    ppo_steps: int
    ppo_rollout: int
    ppo_epochs: int
    td3_steps: int
    sac_steps: int
    warmup_steps: int
    batch_size: int
    ablation_steps: int

FAST_BUDGET = Budget(
    hidden=32,
    n_eval=60,
    seeds=[0],
    sequential_path_len=1500,
    ppo_steps=768,
    ppo_rollout=128,
    ppo_epochs=2,
    td3_steps=900,
    sac_steps=900,
    warmup_steps=150,
    batch_size=64,
    ablation_steps=768,
)

FULL_BUDGET = Budget(
    hidden=64,
    n_eval=1000,
    seeds=[0, 1, 2],
    sequential_path_len=100_000,
    ppo_steps=60_000,
    ppo_rollout=2048,
    ppo_epochs=10,
    td3_steps=60_000,
    sac_steps=60_000,
    warmup_steps=2000,
    batch_size=256,
    ablation_steps=40_000,
)

base_cfg = MarketConfig()
budget = FULL_BUDGET if RUN_FULL else FAST_BUDGET

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(0)
print("RUN_FULL =", RUN_FULL)
print("base_cfg =", base_cfg)
print("budget =", budget)


## 1. RL 在这个作业里的最小直觉

强化学习通常由 Markov Decision Process 描述：

- **状态 \(s_t\)**：当前可观察信息。这里是 \((\log S_t,L_t)\)，Reward (2) 还要加 \(A_{t-1}\)。
- **动作 \(a_t\)**：智能体选择什么。这里是持仓 \(A_t\)，连续且可为负，负数代表做空。
- **奖励 \(r_t\)**：动作之后得到的即时收益。这里既有价格变动收益，也有仓位或换手惩罚。
- **策略 \(\pi(a|s)\)**：从状态到动作的规则。PPO 学 stochastic policy，TD3 学 deterministic policy，SAC 学 entropy-regularized stochastic policy。
- **价值函数 \(V(s)\) 或 \(Q(s,a)\)**：从当前状态或状态-动作出发，未来折扣收益的期望。

本作业最重要的实践点是：金融 RL 不能只看训练 reward。最终必须用独立 Monte Carlo 模拟路径评估 learned policy，避免把训练路径或采样噪声当成真实泛化能力。


## 2. 作业模型与两个数据协议

价格过程：

\[
L_{t+1}=(1-\kappa)L_t+\sigma Z_t,\qquad Z_t\sim N(0,1)
\]

\[
S_{t+1}=S_t e^{L_{t+1}},\qquad S_0=1,\quad L_0=0
\]

两种 reward：

\[
R^{(1)}_t=A_t(S_{t+1}-S_t)-\lambda A_t^2
\]

\[
R^{(2)}_t=A_t(S_{t+1}-S_t)-\lambda(A_t-A_{t-1})^2S_t,\qquad A_{-1}=0
\]

两个数据 setting：

- **Simulation oracle**：训练时可以不断 reset，并从模型生成新的独立 episode。
- **Sequential data**：训练数据来自一条固定长价格路径，不能任意生成新的独立训练路径。对 off-policy 算法，本 notebook 用 behavior policy 先收集 fixed replay buffer，再离线更新 TD3/SAC。


In [ ]:
class MeanRevertingStockEnv:
    '''Oracle environment for the assignment's mean-reverting stock model.'''

    def __init__(self, cfg: MarketConfig, reward_type: str = "R1", seed: int = 0):
        assert reward_type in {"R1", "R2"}
        self.cfg = cfg
        self.reward_type = reward_type
        self.rng = np.random.default_rng(seed)
        self.reset()

    @property
    def state_dim(self) -> int:
        return 2 if self.reward_type == "R1" else 3

    def _state(self) -> np.ndarray:
        state = [self.log_s, self.L]
        if self.reward_type == "R2":
            state.append(self.a_prev)
        return np.array(state, dtype=np.float32)

    def reset(self) -> np.ndarray:
        self.t = 0
        self.log_s = 0.0
        self.S = 1.0
        self.L = 0.0
        self.a_prev = 0.0
        return self._state()

    def step(self, action: float) -> Tuple[np.ndarray, float, bool, Dict]:
        cfg = self.cfg
        A = float(np.clip(action, -cfg.a_max, cfg.a_max))

        L_next = (1.0 - cfg.kappa) * self.L + cfg.sigma * self.rng.normal()
        log_s_next = self.log_s + L_next
        if cfg.log_s_clip is not None:
            log_s_next = float(np.clip(log_s_next, -cfg.log_s_clip, cfg.log_s_clip))
            L_next = log_s_next - self.log_s
        S_next = float(np.exp(log_s_next))

        pnl = A * (S_next - self.S)
        if self.reward_type == "R1":
            reward = pnl - cfg.lam * A**2
        else:
            reward = pnl - cfg.lam * (A - self.a_prev) ** 2 * self.S

        info = {
            "S_t": self.S,
            "S_next": S_next,
            "L_next": L_next,
            "action": A,
            "pnl": pnl,
            "a_prev": self.a_prev,
        }
        self.log_s = log_s_next
        self.S = S_next
        self.L = L_next
        self.a_prev = A
        self.t += 1
        done = self.t >= cfg.T
        return self._state(), float(reward), done, info


def simulate_price_path(cfg: MarketConfig, length: int, seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    log_s = np.zeros(length + 1, dtype=np.float64)
    L = np.zeros(length + 1, dtype=np.float64)
    S = np.ones(length + 1, dtype=np.float64)
    for t in range(length):
        L[t + 1] = (1.0 - cfg.kappa) * L[t] + cfg.sigma * rng.normal()
        log_s[t + 1] = log_s[t] + L[t + 1]
        if cfg.log_s_clip is not None:
            log_s[t + 1] = np.clip(log_s[t + 1], -cfg.log_s_clip, cfg.log_s_clip)
            L[t + 1] = log_s[t + 1] - log_s[t]
        S[t + 1] = np.exp(log_s[t + 1])
    return pd.DataFrame({"t": np.arange(length + 1), "log_s": log_s, "L": L, "S": S})


class SequentialTradingEnv:
    '''Trading environment over a fixed pre-generated price path.'''

    def __init__(self, cfg: MarketConfig, path: pd.DataFrame, reward_type: str = "R1"):
        assert reward_type in {"R1", "R2"}
        self.cfg = cfg
        self.path = path.reset_index(drop=True)
        self.reward_type = reward_type
        self.max_t = len(self.path) - 1
        self.reset()

    @property
    def state_dim(self) -> int:
        return 2 if self.reward_type == "R1" else 3

    def _state(self) -> np.ndarray:
        row = self.path.iloc[self.idx]
        state = [float(row.log_s), float(row.L)]
        if self.reward_type == "R2":
            state.append(self.a_prev)
        return np.array(state, dtype=np.float32)

    def reset(self) -> np.ndarray:
        self.idx = 0
        self.a_prev = 0.0
        return self._state()

    def step(self, action: float) -> Tuple[np.ndarray, float, bool, Dict]:
        cfg = self.cfg
        A = float(np.clip(action, -cfg.a_max, cfg.a_max))
        row = self.path.iloc[self.idx]
        row_next = self.path.iloc[self.idx + 1]
        S_t = float(row.S)
        S_next = float(row_next.S)

        pnl = A * (S_next - S_t)
        if self.reward_type == "R1":
            reward = pnl - cfg.lam * A**2
        else:
            reward = pnl - cfg.lam * (A - self.a_prev) ** 2 * S_t

        info = {"S_t": S_t, "S_next": S_next, "action": A, "a_prev": self.a_prev}
        self.a_prev = A
        self.idx += 1
        done = self.idx >= min(cfg.T, self.max_t)
        return self._state(), float(reward), done, info


def behavior_policy(state: np.ndarray, cfg: MarketConfig, rng: np.random.Generator) -> float:
    '''Default sequential-data behavior policy for off-policy buffers.'''
    L_t = float(state[1])
    action = 2.0 * L_t + rng.normal(0.0, 1.0)
    return float(np.clip(action, -cfg.a_max, cfg.a_max))


## 3. 环境 sanity checks

先不训练 RL，先确认环境本身符合题目：

- `L_t` 是 AR(1) mean-reverting。
- `S_{t+1}=S_t\exp(L_{t+1})`。
- Reward (2) 的状态包含上一期动作。
- 所有动作都会 clip 到 \([-A_{\max}, A_{\max}]\)。


In [ ]:
def random_policy_factory(cfg: MarketConfig, seed: int = 0) -> Callable[[np.ndarray], float]:
    rng = np.random.default_rng(seed)
    return lambda state: float(rng.uniform(-cfg.a_max, cfg.a_max))


def rollout_dataframe(env, policy_fn: Callable[[np.ndarray], float]) -> pd.DataFrame:
    state = env.reset()
    rows = []
    done = False
    while not done:
        action = policy_fn(state)
        next_state, reward, done, info = env.step(action)
        rows.append({
            "t": len(rows),
            "log_s": float(state[0]),
            "L": float(state[1]),
            "action": float(info["action"]),
            "reward": reward,
            "S_t": info["S_t"],
            "S_next": info["S_next"],
        })
        state = next_state
    return pd.DataFrame(rows)


env_check = MeanRevertingStockEnv(base_cfg, "R2", seed=7)
df_check = rollout_dataframe(env_check, random_policy_factory(base_cfg, seed=8))

assert len(df_check) == base_cfg.T
assert df_check["action"].abs().max() <= base_cfg.a_max + 1e-8
assert np.isfinite(df_check[["S_t", "S_next", "reward"]].to_numpy()).all()
print("Sanity checks passed. Example trajectory head:")
display(df_check.head())

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].plot(df_check["t"], df_check["S_t"])
axes[0].set(title="Sample price path", xlabel="t", ylabel="S_t")
axes[1].plot(df_check["t"], df_check["L"])
axes[1].axhline(0, color="black", lw=0.8, ls="--")
axes[1].set(title="Mean-reverting L_t", xlabel="t", ylabel="L_t")
axes[2].plot(df_check["t"], df_check["action"])
axes[2].set(title="Random actions", xlabel="t", ylabel="A_t")
plt.tight_layout()
plt.savefig("fig_sanity_trajectory.png", bbox_inches="tight")
plt.show()


## 4. Reward (1) 的解析最优策略

Reward (1) 是：

\[
R_t=A_t(S_{t+1}-S_t)-\lambda A_t^2
\]

关键观察：\(A_t\) 不影响未来价格过程。因此 Bellman 最优动作可以逐期解耦：

\[
\max_{A_t}\; A_t\,\mathbb{E}_t[S_{t+1}-S_t]-\lambda A_t^2
\]

由于

\[
L_{t+1}|L_t\sim N((1-\kappa)L_t,\sigma^2)
\]

所以

\[
\mathbb{E}_t[S_{t+1}]=S_t\exp((1-\kappa)L_t+\tfrac12\sigma^2)
\]

一阶条件给出：

\[
A_t^*=\frac{S_t(\exp((1-\kappa)L_t+\tfrac12\sigma^2)-1)}{2\lambda}
\]

实际实验中还会 clip 到动作范围。


In [ ]:
def analytic_policy_r1(state: np.ndarray, cfg: MarketConfig) -> float:
    log_s, L_t = float(state[0]), float(state[1])
    S_t = math.exp(log_s)
    expected_delta_s = S_t * (math.exp((1.0 - cfg.kappa) * L_t + 0.5 * cfg.sigma**2) - 1.0)
    action = expected_delta_s / (2.0 * cfg.lam)
    return float(np.clip(action, -cfg.a_max, cfg.a_max))


def evaluate_policy_mc(
    cfg: MarketConfig,
    reward_type: str,
    policy_fn: Callable[[np.ndarray], float],
    n_eval: int,
    seed: int = 0,
) -> Dict[str, float]:
    returns, turnovers, abs_positions = [], [], []
    for i in range(n_eval):
        env = MeanRevertingStockEnv(cfg, reward_type=reward_type, seed=seed + 10_000 * i)
        state = env.reset()
        total = 0.0
        actions = []
        a_prev = 0.0
        for t in range(cfg.T):
            action = float(np.clip(policy_fn(state), -cfg.a_max, cfg.a_max))
            state, reward, done, _ = env.step(action)
            total += (cfg.gamma ** t) * reward
            actions.append(action)
            a_prev = action
            if done:
                break
        arr_actions = np.array(actions, dtype=np.float64)
        prev = np.r_[0.0, arr_actions[:-1]]
        returns.append(total)
        turnovers.append(float(np.mean(np.abs(arr_actions - prev))))
        abs_positions.append(float(np.mean(np.abs(arr_actions))))
    arr = np.array(returns, dtype=np.float64)
    return {
        "mean_return": float(arr.mean()),
        "std_error": float(arr.std(ddof=1) / math.sqrt(len(arr))) if len(arr) > 1 else 0.0,
        "return_std": float(arr.std(ddof=1)) if len(arr) > 1 else 0.0,
        "turnover": float(np.mean(turnovers)),
        "avg_abs_position": float(np.mean(abs_positions)),
        "num_eval_paths": int(n_eval),
        "T_eval": int(cfg.T),
    }


def policy_mse_reward1(cfg: MarketConfig, policy_fn: Callable[[np.ndarray], float]) -> float:
    L_grid = np.linspace(-0.5, 0.5, 101)
    sq = []
    for L_t in L_grid:
        state = np.array([0.0, L_t], dtype=np.float32)
        sq.append((policy_fn(state) - analytic_policy_r1(state, cfg)) ** 2)
    return float(np.mean(sq))


random_eval = evaluate_policy_mc(base_cfg, "R1", random_policy_factory(base_cfg, 1), budget.n_eval, seed=100)
analytic_eval = evaluate_policy_mc(base_cfg, "R1", lambda s: analytic_policy_r1(s, base_cfg), budget.n_eval, seed=200)
baseline_df = pd.DataFrame([
    {"policy": "random", **random_eval},
    {"policy": "analytic_R1", **analytic_eval},
])
display(baseline_df)
assert analytic_eval["mean_return"] > random_eval["mean_return"], "Analytic R1 should beat random in this sanity check."


## 5. 算法理解：PPO、TD3、SAC

### PPO：on-policy

PPO 用当前策略收集 rollout，然后用 clipped objective 防止策略一步变化过大：

\[
L^{CLIP}=\mathbb{E}[\min(r_t\hat A_t,\text{clip}(r_t,1-\epsilon,1+\epsilon)\hat A_t)]
\]

它稳定，但样本效率通常不如 off-policy。

### TD3：off-policy deterministic actor-critic

TD3 用 replay buffer、双 Q 网络、target policy smoothing 和 delayed actor update 来降低 Q-learning 的过估计和不稳定。

### SAC：off-policy stochastic actor-critic

SAC 最大化 reward 加 entropy：

\[
\mathbb{E}\sum_t \gamma^t (r_t+\alpha H(\pi(\cdot|s_t)))
\]

它鼓励探索，常比 deterministic 方法更稳，但实现更复杂。


In [ ]:
class ReplayBuffer:
    def __init__(self, capacity: int = 200_000):
        self.capacity = capacity
        self.storage = []
        self.pos = 0

    def add(self, state, action, reward, next_state, done):
        item = (
            np.asarray(state, dtype=np.float32),
            np.asarray([action], dtype=np.float32),
            np.asarray([reward], dtype=np.float32),
            np.asarray(next_state, dtype=np.float32),
            np.asarray([done], dtype=np.float32),
        )
        if len(self.storage) < self.capacity:
            self.storage.append(item)
        else:
            self.storage[self.pos] = item
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size: int):
        idx = np.random.randint(0, len(self.storage), size=batch_size)
        batch = [self.storage[i] for i in idx]
        s, a, r, ns, d = zip(*batch)
        return (
            torch.as_tensor(np.array(s), dtype=torch.float32, device=DEVICE),
            torch.as_tensor(np.array(a), dtype=torch.float32, device=DEVICE),
            torch.as_tensor(np.array(r), dtype=torch.float32, device=DEVICE),
            torch.as_tensor(np.array(ns), dtype=torch.float32, device=DEVICE),
            torch.as_tensor(np.array(d), dtype=torch.float32, device=DEVICE),
        )

    def __len__(self):
        return len(self.storage)


def mlp(input_dim: int, output_dim: int, hidden: int, activation=nn.ReLU) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(input_dim, hidden), activation(),
        nn.Linear(hidden, hidden), activation(),
        nn.Linear(hidden, output_dim),
    )


class PPOActor(nn.Module):
    def __init__(self, state_dim: int, hidden: int, a_max: float):
        super().__init__()
        self.a_max = a_max
        self.net = mlp(state_dim, 1, hidden, activation=nn.Tanh)
        self.log_std = nn.Parameter(torch.tensor([-0.5], dtype=torch.float32))

    def distribution(self, states: torch.Tensor) -> Normal:
        mean = torch.tanh(self.net(states)) * self.a_max
        std = torch.exp(self.log_std).expand_as(mean)
        return Normal(mean, std)

    def act(self, state: np.ndarray, deterministic: bool = False) -> float:
        with torch.no_grad():
            s = torch.as_tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            dist = self.distribution(s)
            action = dist.mean if deterministic else dist.sample()
        return float(torch.clamp(action, -self.a_max, self.a_max).cpu().item())


class ValueNet(nn.Module):
    def __init__(self, state_dim: int, hidden: int):
        super().__init__()
        self.net = mlp(state_dim, 1, hidden, activation=nn.Tanh)

    def forward(self, states: torch.Tensor) -> torch.Tensor:
        return self.net(states).squeeze(-1)


class DeterministicActor(nn.Module):
    def __init__(self, state_dim: int, hidden: int, a_max: float):
        super().__init__()
        self.a_max = a_max
        self.net = mlp(state_dim, 1, hidden, activation=nn.ReLU)

    def forward(self, states: torch.Tensor) -> torch.Tensor:
        return torch.tanh(self.net(states)) * self.a_max

    def act(self, state: np.ndarray) -> float:
        with torch.no_grad():
            s = torch.as_tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            return float(self.forward(s).cpu().item())


class QNet(nn.Module):
    def __init__(self, state_dim: int, hidden: int):
        super().__init__()
        self.net = mlp(state_dim + 1, 1, hidden, activation=nn.ReLU)

    def forward(self, states: torch.Tensor, actions: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([states, actions], dim=-1))


class SACActor(nn.Module):
    LOG_STD_MIN = -5.0
    LOG_STD_MAX = 2.0

    def __init__(self, state_dim: int, hidden: int, a_max: float):
        super().__init__()
        self.a_max = a_max
        self.trunk = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.mean = nn.Linear(hidden, 1)
        self.log_std = nn.Linear(hidden, 1)

    def _params(self, states: torch.Tensor):
        h = self.trunk(states)
        mean = self.mean(h)
        log_std = self.log_std(h).clamp(self.LOG_STD_MIN, self.LOG_STD_MAX)
        return mean, log_std

    def sample(self, states: torch.Tensor):
        mean, log_std = self._params(states)
        std = log_std.exp()
        normal = Normal(mean, std)
        z = normal.rsample()
        tanh_z = torch.tanh(z)
        action = tanh_z * self.a_max
        log_prob = normal.log_prob(z) - torch.log(self.a_max * (1.0 - tanh_z.pow(2)) + 1e-6)
        return action, log_prob.sum(dim=-1, keepdim=True)

    def act(self, state: np.ndarray, deterministic: bool = True) -> float:
        with torch.no_grad():
            s = torch.as_tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            if deterministic:
                mean, _ = self._params(s)
                action = torch.tanh(mean) * self.a_max
            else:
                action, _ = self.sample(s)
        return float(action.cpu().item())


def soft_update(net: nn.Module, target: nn.Module, tau: float) -> None:
    for p, tp in zip(net.parameters(), target.parameters()):
        tp.data.mul_(1.0 - tau).add_(tau * p.data)


## 6. PPO implementation

这个实现是教学版 PPO：
- Gaussian actor 输出动作均值。
- critic 估计 \(V(s)\)。
- GAE 估计 advantage。
- 动作会 clip 到 \([-A_{\max},A_{\max}]\)。

严格的 bounded stochastic policy 会使用 tanh-squashed Gaussian；这里为了让代码更适合学习，PPO 用 normal sampling + action clipping。


In [ ]:
def compute_gae(
    rewards: List[float],
    values: List[float],
    dones: List[bool],
    last_value: float,
    gamma: float,
    gae_lambda: float = 0.95,
):
    adv = np.zeros(len(rewards), dtype=np.float32)
    gae = 0.0
    for t in reversed(range(len(rewards))):
        next_value = last_value if t == len(rewards) - 1 else values[t + 1]
        mask = 1.0 - float(dones[t])
        delta = rewards[t] + gamma * next_value * mask - values[t]
        gae = delta + gamma * gae_lambda * mask * gae
        adv[t] = gae
    returns = adv + np.asarray(values, dtype=np.float32)
    return adv, returns


def make_env_for_setting(cfg: MarketConfig, reward_type: str, setting: str, seed: int):
    if setting == "oracle":
        return MeanRevertingStockEnv(cfg, reward_type=reward_type, seed=seed)
    path = simulate_price_path(cfg, budget.sequential_path_len, seed=seed)
    return SequentialTradingEnv(cfg, path, reward_type=reward_type)


def train_ppo(
    cfg: MarketConfig,
    reward_type: str,
    setting: str,
    seed: int,
    lr: float = 3e-4,
    total_steps: Optional[int] = None,
):
    set_seed(seed)
    total_steps = budget.ppo_steps if total_steps is None else total_steps
    env = make_env_for_setting(cfg, reward_type, setting, seed=seed + 11)
    actor = PPOActor(env.state_dim, budget.hidden, cfg.a_max).to(DEVICE)
    critic = ValueNet(env.state_dim, budget.hidden).to(DEVICE)
    opt = torch.optim.Adam(list(actor.parameters()) + list(critic.parameters()), lr=lr)

    state = env.reset()
    ep_return = 0.0
    recent = deque(maxlen=10)
    log_rows = []
    steps_done = 0

    while steps_done < total_steps:
        s_buf, raw_a_buf, logp_buf, r_buf, d_buf, v_buf = [], [], [], [], [], []
        for _ in range(budget.ppo_rollout):
            st = torch.as_tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            with torch.no_grad():
                dist = actor.distribution(st)
                raw_action = dist.sample()
                logp = dist.log_prob(raw_action).sum(dim=-1)
                value = critic(st)
            clipped_action = float(torch.clamp(raw_action, -cfg.a_max, cfg.a_max).cpu().item())
            next_state, reward, done, _ = env.step(clipped_action)

            s_buf.append(state.copy())
            raw_a_buf.append([float(raw_action.cpu().item())])
            logp_buf.append(float(logp.cpu().item()))
            r_buf.append(float(reward))
            d_buf.append(bool(done))
            v_buf.append(float(value.cpu().item()))

            ep_return += reward
            state = next_state
            steps_done += 1
            if done:
                recent.append(ep_return)
                ep_return = 0.0
                state = env.reset()
            if steps_done >= total_steps:
                break

        with torch.no_grad():
            last_value = float(critic(torch.as_tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)).cpu().item())
        adv, ret = compute_gae(r_buf, v_buf, d_buf, last_value, cfg.gamma)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        s_t = torch.as_tensor(np.asarray(s_buf), dtype=torch.float32, device=DEVICE)
        a_t = torch.as_tensor(np.asarray(raw_a_buf), dtype=torch.float32, device=DEVICE)
        old_logp_t = torch.as_tensor(np.asarray(logp_buf), dtype=torch.float32, device=DEVICE)
        adv_t = torch.as_tensor(adv, dtype=torch.float32, device=DEVICE)
        ret_t = torch.as_tensor(ret, dtype=torch.float32, device=DEVICE)

        n = len(s_buf)
        for _ in range(budget.ppo_epochs):
            perm = np.random.permutation(n)
            for start in range(0, n, budget.batch_size):
                idx = perm[start:start + budget.batch_size]
                dist = actor.distribution(s_t[idx])
                new_logp = dist.log_prob(a_t[idx]).sum(dim=-1)
                ratio = torch.exp(new_logp - old_logp_t[idx])
                unclipped = ratio * adv_t[idx]
                clipped = torch.clamp(ratio, 0.8, 1.2) * adv_t[idx]
                actor_loss = -torch.min(unclipped, clipped).mean()
                value_loss = F.mse_loss(critic(s_t[idx]), ret_t[idx])
                entropy_bonus = dist.entropy().sum(dim=-1).mean()
                loss = actor_loss + 0.5 * value_loss - 0.001 * entropy_bonus

                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(list(actor.parameters()) + list(critic.parameters()), 0.5)
                opt.step()

        if recent:
            log_rows.append({"steps": steps_done, "recent_train_return": float(np.mean(recent))})

    return {"algo": "PPO", "actor": actor, "critic": critic}, pd.DataFrame(log_rows)


## 7. TD3 implementation

TD3 是 off-policy 方法。Oracle setting 下，actor 和环境交互并写入 replay buffer；sequential setting 下，先用 behavior policy 在固定路径上收集 buffer，然后 TD3 只从这个 buffer 更新。


In [ ]:
def build_behavior_buffer(cfg: MarketConfig, reward_type: str, seed: int, n_steps: int) -> ReplayBuffer:
    rng = np.random.default_rng(seed)
    path = simulate_price_path(cfg, max(n_steps + 2, cfg.T + 2), seed=seed + 101)
    env = SequentialTradingEnv(cfg, path, reward_type=reward_type)
    buffer = ReplayBuffer(capacity=n_steps + 10)
    state = env.reset()
    for _ in range(n_steps):
        action = behavior_policy(state, cfg, rng)
        next_state, reward, done, _ = env.step(action)
        buffer.add(state, action, reward, next_state, done)
        state = next_state
        if done:
            state = env.reset()
    return buffer


def train_td3(
    cfg: MarketConfig,
    reward_type: str,
    setting: str,
    seed: int,
    actor_lr: float = 1e-3,
    critic_lr: float = 1e-3,
    total_steps: Optional[int] = None,
):
    set_seed(seed)
    total_steps = budget.td3_steps if total_steps is None else total_steps
    state_dim = 2 if reward_type == "R1" else 3
    actor = DeterministicActor(state_dim, budget.hidden, cfg.a_max).to(DEVICE)
    actor_tgt = DeterministicActor(state_dim, budget.hidden, cfg.a_max).to(DEVICE)
    q1, q2 = QNet(state_dim, budget.hidden).to(DEVICE), QNet(state_dim, budget.hidden).to(DEVICE)
    q1_tgt, q2_tgt = QNet(state_dim, budget.hidden).to(DEVICE), QNet(state_dim, budget.hidden).to(DEVICE)
    actor_tgt.load_state_dict(actor.state_dict())
    q1_tgt.load_state_dict(q1.state_dict())
    q2_tgt.load_state_dict(q2.state_dict())
    actor_opt = torch.optim.Adam(actor.parameters(), lr=actor_lr)
    critic_opt = torch.optim.Adam(list(q1.parameters()) + list(q2.parameters()), lr=critic_lr)

    if setting == "sequential":
        buffer = build_behavior_buffer(cfg, reward_type, seed=seed + 33, n_steps=max(total_steps, 500))
        env = None
        state = None
    else:
        buffer = ReplayBuffer(capacity=200_000)
        env = MeanRevertingStockEnv(cfg, reward_type=reward_type, seed=seed + 33)
        state = env.reset()

    log_rows = []
    ep_return = 0.0
    recent = deque(maxlen=10)
    tau = 0.005
    policy_delay = 2
    target_noise = 0.2 * cfg.a_max
    noise_clip = 0.5 * cfg.a_max
    exploration_noise = 0.15 * cfg.a_max

    for step in range(1, total_steps + 1):
        if setting == "oracle":
            if step <= budget.warmup_steps:
                action = np.random.uniform(-cfg.a_max, cfg.a_max)
            else:
                action = actor.act(state) + np.random.normal(0.0, exploration_noise)
            action = float(np.clip(action, -cfg.a_max, cfg.a_max))
            next_state, reward, done, _ = env.step(action)
            buffer.add(state, action, reward, next_state, done)
            state = next_state
            ep_return += reward
            if done:
                recent.append(ep_return)
                ep_return = 0.0
                state = env.reset()

        if len(buffer) >= budget.batch_size:
            s, a, r, ns, d = buffer.sample(budget.batch_size)
            with torch.no_grad():
                noise = torch.randn_like(a) * target_noise
                noise = torch.clamp(noise, -noise_clip, noise_clip)
                next_action = torch.clamp(actor_tgt(ns) + noise, -cfg.a_max, cfg.a_max)
                q_target = torch.min(q1_tgt(ns, next_action), q2_tgt(ns, next_action))
                y = r + cfg.gamma * (1.0 - d) * q_target
            critic_loss = F.mse_loss(q1(s, a), y) + F.mse_loss(q2(s, a), y)
            critic_opt.zero_grad()
            critic_loss.backward()
            critic_opt.step()

            if step % policy_delay == 0:
                actor_loss = -q1(s, actor(s)).mean()
                actor_opt.zero_grad()
                actor_loss.backward()
                actor_opt.step()
                soft_update(actor, actor_tgt, tau)
                soft_update(q1, q1_tgt, tau)
                soft_update(q2, q2_tgt, tau)

        if step % max(100, total_steps // 3) == 0:
            log_rows.append({
                "steps": step,
                "recent_train_return": float(np.mean(recent)) if recent else np.nan,
                "buffer_size": len(buffer),
            })

    return {"algo": "TD3", "actor": actor}, pd.DataFrame(log_rows)


## 8. SAC implementation

SAC 也是 off-policy。它和 TD3 一样使用 replay buffer，但 actor 是 stochastic，并且目标里加入 entropy 项。这里实现自动温度 \(\alpha\) 更新。


In [ ]:
def train_sac(
    cfg: MarketConfig,
    reward_type: str,
    setting: str,
    seed: int,
    lr: float = 3e-4,
    total_steps: Optional[int] = None,
):
    set_seed(seed)
    total_steps = budget.sac_steps if total_steps is None else total_steps
    state_dim = 2 if reward_type == "R1" else 3
    actor = SACActor(state_dim, budget.hidden, cfg.a_max).to(DEVICE)
    q1, q2 = QNet(state_dim, budget.hidden).to(DEVICE), QNet(state_dim, budget.hidden).to(DEVICE)
    q1_tgt, q2_tgt = QNet(state_dim, budget.hidden).to(DEVICE), QNet(state_dim, budget.hidden).to(DEVICE)
    q1_tgt.load_state_dict(q1.state_dict())
    q2_tgt.load_state_dict(q2.state_dict())
    actor_opt = torch.optim.Adam(actor.parameters(), lr=lr)
    q_opt = torch.optim.Adam(list(q1.parameters()) + list(q2.parameters()), lr=lr)
    log_alpha = torch.tensor(np.log(0.2), dtype=torch.float32, requires_grad=True, device=DEVICE)
    alpha_opt = torch.optim.Adam([log_alpha], lr=lr)
    target_entropy = -1.0

    if setting == "sequential":
        buffer = build_behavior_buffer(cfg, reward_type, seed=seed + 44, n_steps=max(total_steps, 500))
        env = None
        state = None
    else:
        buffer = ReplayBuffer(capacity=200_000)
        env = MeanRevertingStockEnv(cfg, reward_type=reward_type, seed=seed + 44)
        state = env.reset()

    log_rows = []
    ep_return = 0.0
    recent = deque(maxlen=10)
    tau = 0.005

    for step in range(1, total_steps + 1):
        if setting == "oracle":
            if step <= budget.warmup_steps:
                action = np.random.uniform(-cfg.a_max, cfg.a_max)
            else:
                action = actor.act(state, deterministic=False)
            next_state, reward, done, _ = env.step(action)
            buffer.add(state, action, reward, next_state, done)
            state = next_state
            ep_return += reward
            if done:
                recent.append(ep_return)
                ep_return = 0.0
                state = env.reset()

        if len(buffer) >= budget.batch_size:
            s, a, r, ns, d = buffer.sample(budget.batch_size)
            alpha = log_alpha.exp().detach()
            with torch.no_grad():
                next_a, next_logp = actor.sample(ns)
                next_q = torch.min(q1_tgt(ns, next_a), q2_tgt(ns, next_a)) - alpha * next_logp
                y = r + cfg.gamma * (1.0 - d) * next_q
            q_loss = F.mse_loss(q1(s, a), y) + F.mse_loss(q2(s, a), y)
            q_opt.zero_grad()
            q_loss.backward()
            q_opt.step()

            new_a, logp = actor.sample(s)
            q_new = torch.min(q1(s, new_a), q2(s, new_a))
            actor_loss = (log_alpha.exp().detach() * logp - q_new).mean()
            actor_opt.zero_grad()
            actor_loss.backward()
            actor_opt.step()

            alpha_loss = -(log_alpha * (logp.detach() + target_entropy)).mean()
            alpha_opt.zero_grad()
            alpha_loss.backward()
            alpha_opt.step()

            soft_update(q1, q1_tgt, tau)
            soft_update(q2, q2_tgt, tau)

        if step % max(100, total_steps // 3) == 0:
            log_rows.append({
                "steps": step,
                "recent_train_return": float(np.mean(recent)) if recent else np.nan,
                "buffer_size": len(buffer),
                "alpha": float(log_alpha.exp().detach().cpu().item()),
            })

    return {"algo": "SAC", "actor": actor}, pd.DataFrame(log_rows)


## 9. 统一训练与评估接口

下面的实验矩阵是：

- rewards: `R1`, `R2`
- settings: `oracle`, `sequential`
- algorithms: `PPO`, `TD3`, `SAC`

在 fast mode 下，这只是 smoke training，不能把数值当最终结论。它的价值是确认所有路径能跑通，并生成报告所需的表格和图。


In [ ]:
def policy_from_agent(agent: Dict) -> Callable[[np.ndarray], float]:
    algo = agent["algo"]
    actor = agent["actor"]
    if algo == "PPO":
        return lambda state: actor.act(state, deterministic=True)
    if algo == "TD3":
        return lambda state: actor.act(state)
    if algo == "SAC":
        return lambda state: actor.act(state, deterministic=True)
    raise ValueError(algo)


def train_agent(algo: str, cfg: MarketConfig, reward_type: str, setting: str, seed: int, **kwargs):
    if algo == "PPO":
        return train_ppo(cfg, reward_type, setting, seed, **kwargs)
    if algo == "TD3":
        return train_td3(cfg, reward_type, setting, seed, **kwargs)
    if algo == "SAC":
        return train_sac(cfg, reward_type, setting, seed, **kwargs)
    raise ValueError(algo)


def parameter_sets(run_full: bool) -> List[Tuple[str, MarketConfig]]:
    if not run_full:
        return [("baseline", base_cfg)]
    return [
        ("baseline", base_cfg),
        ("slow_reversion", replace(base_cfg, kappa=0.05)),
        ("fast_reversion", replace(base_cfg, kappa=0.5)),
        ("high_volatility", replace(base_cfg, sigma=0.2)),
        ("strong_regularization", replace(base_cfg, lam=0.05)),
        ("high_discount", replace(base_cfg, gamma=0.99, T=200)),
    ]


agents: Dict[Tuple[str, str, str, int], Dict] = {}
logs: Dict[Tuple[str, str, str, int], pd.DataFrame] = {}
result_rows = []

if RUN_MAIN_MATRIX:
    for param_label, cfg_exp in parameter_sets(RUN_FULL):
        print(f"\n=== Parameter set: {param_label} ===")
        analytic_metrics = evaluate_policy_mc(
            cfg_exp, "R1", lambda s, c=cfg_exp: analytic_policy_r1(s, c),
            n_eval=budget.n_eval, seed=90_000
        )
        result_rows.append({
            "parameter_set": param_label,
            "reward_type": "R1",
            "method": "Analytic",
            "train_setting": "none",
            "seed": -1,
            **analytic_metrics,
            "relative_to_analytic_pct": 100.0,
            "regret": 0.0,
            "policy_mse_to_analytic": 0.0,
        })

        for seed in budget.seeds:
            for reward_type in ["R1", "R2"]:
                for setting in ["oracle", "sequential"]:
                    for algo in ["PPO", "TD3", "SAC"]:
                        print(f"Training {algo:>3} | {reward_type} | {setting:10s} | seed={seed}")
                        agent, log_df = train_agent(algo, cfg_exp, reward_type, setting, seed=seed)
                        key = (algo, reward_type, setting, seed)
                        agents[key] = agent
                        logs[key] = log_df
                        policy_fn = policy_from_agent(agent)
                        metrics = evaluate_policy_mc(
                            cfg_exp, reward_type, policy_fn,
                            n_eval=budget.n_eval, seed=30_000 + 1000 * seed
                        )
                        row = {
                            "parameter_set": param_label,
                            "reward_type": reward_type,
                            "method": algo,
                            "train_setting": setting,
                            "seed": seed,
                            **metrics,
                            "relative_to_analytic_pct": np.nan,
                            "regret": np.nan,
                            "policy_mse_to_analytic": np.nan,
                        }
                        if reward_type == "R1":
                            row["relative_to_analytic_pct"] = 100.0 * metrics["mean_return"] / analytic_metrics["mean_return"]
                            row["regret"] = analytic_metrics["mean_return"] - metrics["mean_return"]
                            row["policy_mse_to_analytic"] = policy_mse_reward1(cfg_exp, policy_fn)
                        result_rows.append(row)

results_main = pd.DataFrame(result_rows)
results_main.to_csv("results_main.csv", index=False)
display(results_main)


## 10. 主结果怎么读

表格里最重要的列：

- `mean_return`: 独立 Monte Carlo evaluation 的平均折扣总收益。
- `std_error`: Monte Carlo 标准误。
- `relative_to_analytic_pct`: 仅 Reward (1) 有意义，解析最优策略是 100%。
- `regret`: 解析最优收益减去 learned policy 收益。
- `turnover`: 平均换手率，Reward (2) 下应重点观察。
- `policy_mse_to_analytic`: Reward (1) learned policy 与解析策略曲线的 MSE。

fast mode 只说明代码路径可跑通；正式报告应使用 full mode 的结果。


In [ ]:
if len(results_main):
    summary_cols = [
        "parameter_set", "reward_type", "method", "train_setting",
        "mean_return", "std_error", "relative_to_analytic_pct",
        "regret", "turnover", "policy_mse_to_analytic",
    ]
    display(results_main[summary_cols].sort_values(["reward_type", "method", "train_setting"]).reset_index(drop=True))


## 11. Policy visualization

可视化策略比只看 reward 更重要。

Reward (1) 有解析策略，所以可以画：

\[
L_t \mapsto A_t \quad\text{at }S_t=1
\]

Reward (2) 没有简单解析策略，所以画：

\[
(L_t,A_{t-1})\mapsto A_t \quad\text{at }S_t=1
\]

如果交易成本有效，Reward (2) 的策略通常会更依赖 \(A_{t-1}\)，换手更平滑。


In [ ]:
def get_agent_or_none(algo: str, reward_type: str, setting: str, seed: int = 0):
    return agents.get((algo, reward_type, setting, seed))


if RUN_MAIN_MATRIX and agents:
    fig, ax = plt.subplots(figsize=(7, 4))
    for key, log_df in logs.items():
        algo, reward_type, setting, seed = key
        if reward_type == "R1" and setting == "oracle" and len(log_df):
            ax.plot(log_df["steps"], log_df["recent_train_return"], label=f"{algo} {setting}")
    ax.set(title="Training curves (R1 oracle, fast smoke)", xlabel="training steps", ylabel="recent train return")
    ax.legend()
    plt.tight_layout()
    plt.savefig("fig_training_curves.png", bbox_inches="tight")
    plt.show()

    L_grid = np.linspace(-0.5, 0.5, 201)
    states_r1 = [np.array([0.0, L], dtype=np.float32) for L in L_grid]
    plt.figure(figsize=(8, 4))
    plt.plot(L_grid, [analytic_policy_r1(s, base_cfg) for s in states_r1], "k-", lw=2, label="Analytic R1")
    for algo, style in [("PPO", "--"), ("TD3", ":"), ("SAC", "-.")]:
        agent = get_agent_or_none(algo, "R1", "oracle", seed=0)
        if agent is not None:
            pol = policy_from_agent(agent)
            plt.plot(L_grid, [pol(s) for s in states_r1], style, lw=1.5, label=f"{algo} oracle")
    plt.axhline(0, color="black", lw=0.6)
    plt.axvline(0, color="black", lw=0.6)
    plt.xlabel("$L_t$")
    plt.ylabel("$A_t$")
    plt.title("Reward (1): learned policies vs analytic policy at $S_t=1$")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig("fig_reward1_policy_comparison.png", bbox_inches="tight")
    plt.show()

    # R1 heatmap: analytic vs one learned policy
    L_vals = np.linspace(-0.4, 0.4, 50)
    log_s_vals = np.linspace(-0.5, 0.5, 50)
    LL, SS = np.meshgrid(L_vals, log_s_vals)

    def grid_eval_r1(policy_fn):
        out = np.zeros_like(LL)
        for i in range(LL.shape[0]):
            for j in range(LL.shape[1]):
                out[i, j] = policy_fn(np.array([SS[i, j], LL[i, j]], dtype=np.float32))
        return out

    ppo_agent = get_agent_or_none("PPO", "R1", "oracle", seed=0)
    if ppo_agent is not None:
        A_analytic = grid_eval_r1(lambda s: analytic_policy_r1(s, base_cfg))
        A_ppo = grid_eval_r1(policy_from_agent(ppo_agent))
        vmax = max(np.abs(A_analytic).max(), np.abs(A_ppo).max())
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        for ax, data, title in zip(axes, [A_analytic, A_ppo], ["Analytic", "PPO oracle"]):
            im = ax.pcolormesh(L_vals, log_s_vals, data, cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto")
            fig.colorbar(im, ax=ax, label="$A_t$")
            ax.set(xlabel="$L_t$", ylabel="$\\log S_t$", title=title)
        plt.tight_layout()
        plt.savefig("fig_reward1_policy_heatmap.png", bbox_inches="tight")
        plt.show()

    # R2 heatmap.
    r2_agent = get_agent_or_none("PPO", "R2", "oracle", seed=0)
    if r2_agent is not None:
        pol_r2 = policy_from_agent(r2_agent)
        L_vals = np.linspace(-0.4, 0.4, 50)
        a_prev_vals = np.linspace(-base_cfg.a_max, base_cfg.a_max, 50)
        LL, AA = np.meshgrid(L_vals, a_prev_vals)
        grid = np.zeros_like(LL)
        for i in range(LL.shape[0]):
            for j in range(LL.shape[1]):
                grid[i, j] = pol_r2(np.array([0.0, LL[i, j], AA[i, j]], dtype=np.float32))
        plt.figure(figsize=(6, 4.5))
        im = plt.pcolormesh(L_vals, a_prev_vals, grid, cmap="RdBu_r", shading="auto")
        plt.colorbar(im, label="$A_t$")
        plt.xlabel("$L_t$")
        plt.ylabel("$A_{t-1}$")
        plt.title("Reward (2): PPO oracle policy heatmap at $S_t=1$")
        plt.tight_layout()
        plt.savefig("fig_reward2_policy_heatmap.png", bbox_inches="tight")
        plt.show()


## 12. Ablation study：learning rate

作业要求至少一个 ablation study。这里固定：

- Reward (1)
- simulation oracle
- PPO
- baseline market parameters

只改变 learning rate，比较 final MC return 和 policy MSE。


In [ ]:
ablation_rows = []
if RUN_ABLATION:
    lr_values = [1e-4, 3e-4, 1e-3] if RUN_FULL else [1e-4, 3e-4]
    for lr in lr_values:
        print(f"Ablation PPO lr={lr}")
        agent_lr, log_lr = train_ppo(
            base_cfg, "R1", "oracle", seed=123,
            lr=lr,
            total_steps=budget.ablation_steps,
        )
        pol = policy_from_agent(agent_lr)
        metrics = evaluate_policy_mc(base_cfg, "R1", pol, n_eval=budget.n_eval, seed=70_000)
        ablation_rows.append({
            "lr": lr,
            **metrics,
            "policy_mse_to_analytic": policy_mse_reward1(base_cfg, pol),
        })
        if len(log_lr):
            plt.plot(log_lr["steps"], log_lr["recent_train_return"], label=f"lr={lr}")
    plt.title("Learning-rate ablation: PPO R1 oracle")
    plt.xlabel("training steps")
    plt.ylabel("recent train return")
    plt.legend()
    plt.tight_layout()
    plt.savefig("fig_ablation_learning_rate.png", bbox_inches="tight")
    plt.show()

results_ablation = pd.DataFrame(ablation_rows)
results_ablation.to_csv("results_ablation.csv", index=False)
display(results_ablation)


## 13. Parameter sensitivity

Full mode 下会对多个参数组重训 PPO/TD3/SAC。Fast mode 不默认重训这些组合，因为运行时间会膨胀。

报告写作时可以重点解释：

- \(\kappa\) 小：均值回复慢，return signal 更持久。
- \(\sigma\) 大：机会和风险同时变大，学习更不稳定。
- \(\lambda\) 大：仓位/换手惩罚更强，策略更保守。
- \(\gamma\) 大：更重视长期，但训练和评估方差更高。


In [ ]:
if RUN_SENSITIVITY:
    sensitivity = results_main.groupby(
        ["parameter_set", "reward_type", "method", "train_setting"], dropna=False
    )[["mean_return", "std_error", "turnover"]].mean().reset_index()
    sensitivity.to_csv("results_sensitivity.csv", index=False)
    display(sensitivity)
else:
    analytic_sensitivity_rows = []
    for label, cfg_exp in parameter_sets(True):
        if label == "high_discount":
            cfg_exp = replace(cfg_exp, T=80 if not RUN_FULL else cfg_exp.T)
        metrics = evaluate_policy_mc(cfg_exp, "R1", lambda s, c=cfg_exp: analytic_policy_r1(s, c), n_eval=budget.n_eval, seed=88_000)
        analytic_sensitivity_rows.append({"parameter_set": label, **metrics})
    analytic_sensitivity = pd.DataFrame(analytic_sensitivity_rows)
    display(analytic_sensitivity[["parameter_set", "mean_return", "std_error", "turnover"]])


## 14. 可直接迁移到 written report 的结构

1. **Introduction**  
   本项目比较 on-policy 与 off-policy RL 方法在模拟均值回复股票交易环境中的表现，并研究 reward design 与 data protocol 对 learned policy 的影响。

2. **Market Environment**  
   写清楚 \(L_{t+1}=(1-\kappa)L_t+\sigma Z_t\)、\(S_{t+1}=S_te^{L_{t+1}}\)、连续动作 \(A_t\)、动作边界和 discounting。

3. **Rewards**  
   Reward (1) 惩罚持仓大小；Reward (2) 惩罚仓位变化，近似 transaction cost。Reward (2) 必须把 \(A_{t-1}\) 放进 state 才满足 Markov property。

4. **Data Settings**  
   Oracle 可以不断生成独立 episode；sequential data 只能沿固定价格路径训练。对 TD3/SAC，sequential setting 需要说明 behavior policy。

5. **Analytic Benchmark**  
   推导 Reward (1) 的闭式最优策略，并说明它是 learned policy 的强 benchmark。

6. **Methods**  
   PPO 使用 on-policy rollout、GAE 和 clipped objective；TD3 使用 replay buffer、twin critics 和 delayed policy update；SAC 使用 entropy-regularized stochastic actor-critic。

7. **Evaluation**  
   所有最终策略用独立模拟路径做 Monte Carlo evaluation，报告 mean discounted truncated total reward 和 standard error。

8. **Results and Discussion**  
   比较 oracle vs sequential、PPO vs TD3/SAC、Reward (1) vs Reward (2)。Reward (1) 重点看 relative performance/regret/policy MSE；Reward (2) 重点看 turnover 和 heatmap smoothness。

9. **Ablation**  
   报告 learning rate 对训练曲线、最终 return 和 policy MSE 的影响。

10. **Conclusion**  
    总结：解析 benchmark 帮助验证 RL 是否学到正确结构；transaction cost 让策略更平滑；sequential data 更接近真实历史数据限制，但覆盖和样本效率更差。


In [ ]:
created_files = [
    "results_main.csv",
    "results_ablation.csv",
    "fig_sanity_trajectory.png",
    "fig_training_curves.png",
    "fig_reward1_policy_comparison.png",
    "fig_reward1_policy_heatmap.png",
    "fig_reward2_policy_heatmap.png",
    "fig_ablation_learning_rate.png",
]
print("Expected output artifacts:")
for f in created_files:
    print(" -", f)
print("\nNotebook finished. For final reported numbers, set RUN_FULL=True and rerun.")
